[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CYJ1/ReACT-TTS/blob/feature/pretrained-encoders-meld-stage-a/notebooks/colab_meld_stage_a.ipynb)

# ReACT-TTS: MELD (dev split) → Stage A response-style predictor

Runs entirely in Colab because this needs real internet access (this dev
session's network policy blocks MELD's distribution hosts and HuggingFace
entirely) plus a free GPU for training. Uses only MELD's **dev split**
(~1,109 utterances, a few hundred MB) instead of the full train split
(~9,989 utterances, several GB) -- enough for a workshop-scale proof of concept.

**Before running**: Runtime → Change runtime type → GPU.

**Honesty note**: MELD's README (checked directly, since raw.githubusercontent.com
happened to be reachable) now points to HuggingFace Datasets
(`declare-lab/MELD`) as the current official source -- the older
`web.eecs.umich.edu` raw-data link some tutorials cite is stale. What was
*not* verified from inside this dev session (huggingface.co itself is
blocked here) is the HF repo's internal file layout, so cell 13 below
prints a directory listing before you set `MELD_CSV`/`MELD_VIDEO_DIR` --
fill those two in based on what you actually see, everything downstream
only depends on those two paths.

## 1. Clone the repo and install dependencies

In [ ]:
!git clone -b feature/pretrained-encoders-meld-stage-a https://github.com/CYJ1/ReACT-TTS.git
%cd ReACT-TTS

In [ ]:
!pip install -q -r requirements.txt
# mediapipe's Tasks API needs EGL/GLES even for CPU-only inference
!apt-get -qq update && apt-get -qq install -y libgl1 libegl1 libgles2 > /dev/null

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('numpy OK:', __import__('numpy').__version__)

## 2. Fetch the pretrained encoder model files
MediaPipe FaceLandmarker (52-dim blendshape expression features) + face detector,
both served from `storage.googleapis.com`. Resemblyzer's speaker-encoder weights
ship inside the pip package already (no separate download).

In [ ]:
!python -m preprocessing.download_pretrained_assets

## 3. (Recommended) Mount Google Drive
Colab sessions are ephemeral / time-limited. Save the downloaded raw data and
the preprocessed manifest+features to Drive so you don't redo this every session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/react_tts_meld'  # change if you like
import os
os.makedirs(DATA_ROOT, exist_ok=True)

## 4. Download MELD (dev split only)

Using only the **dev** split (~1,109 utterances) rather than train (~9,989) --
small enough for a workshop-scale run, same label distribution.

MELD's README now points to HuggingFace Datasets as the current official source
(the older `web.eecs.umich.edu` raw-data link referenced in some tutorials/older README
sections is stale/unreachable): https://huggingface.co/datasets/declare-lab/MELD

This downloads the whole dataset repo (it's not split by train/dev/test as separate
repos) via `huggingface_hub`, no auth needed for this public dataset.

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import snapshot_download

meld_dir = snapshot_download(repo_id='declare-lab/MELD', repo_type='dataset', local_dir=f'{DATA_ROOT}/MELD_hf')
print(meld_dir)

In [ ]:
# Auto-locate dev_sent_emo.csv instead of assuming where it landed (this repo's actual
# HF layout, confirmed: MELD.Raw/{dev,test,train}.tar.gz + MELD.Raw/{dev,test,train}_sent_emo.csv,
# plus a sibling MELD.Features.Models/ folder -- no single MELD.Raw.tar.gz bundle).
import glob, os

search_roots = [f'{DATA_ROOT}/MELD_hf', '/content/ReACT-TTS', '/content']
MELD_CSV = None
for root in search_roots:
    hits = glob.glob(os.path.join(root, '**', 'dev_sent_emo.csv'), recursive=True)
    if hits:
        MELD_CSV = hits[0]
        break
assert MELD_CSV, f'dev_sent_emo.csv not found under {search_roots} -- did the download cell above finish without errors?'
MELD_RAW_DIR = os.path.dirname(MELD_CSV)
print('MELD_CSV:', MELD_CSV)
print('MELD_RAW_DIR:', MELD_RAW_DIR)
!ls {MELD_RAW_DIR}

The CSV is ready to use directly. The videos are still zipped
(`dev.tar.gz`, not `dev_splits_complete.tar.gz` like the old umich.edu
layout) -- extract it and auto-detect whatever folder name it creates.

In [ ]:
tar_path = os.path.join(MELD_RAW_DIR, 'dev.tar.gz')
assert os.path.exists(tar_path), f'expected {tar_path} -- check the `ls` output above for the actual filename'

before = set(os.listdir(MELD_RAW_DIR))
!tar -xzf {tar_path} -C {MELD_RAW_DIR}
after = set(os.listdir(MELD_RAW_DIR))
new_dirs = [d for d in (after - before) if os.path.isdir(os.path.join(MELD_RAW_DIR, d))]
assert len(new_dirs) == 1, f'expected exactly one new folder from extraction, got {new_dirs} -- inspect {MELD_RAW_DIR} manually'

MELD_VIDEO_DIR = os.path.join(MELD_RAW_DIR, new_dirs[0])
print('MELD_VIDEO_DIR:', MELD_VIDEO_DIR)
!ls {MELD_VIDEO_DIR} | head -5

## 5. Build the dyadic subset manifest (small test run first)

This runs face detection + tracking + blendshape embedding over every kept
video, which is the slow step. Start with `--limit_dialogues 15` to sanity-check
the sample count and paths before running it over the full dev split.

In [ ]:
%cd /content/ReACT-TTS
!python -m preprocessing.build_meld_dyadic_subset \
    --meld_csv {MELD_CSV} \
    --video_dir {MELD_VIDEO_DIR} \
    --out_manifest {DATA_ROOT}/manifests/dev_test.jsonl \
    --limit_dialogues 15

In [ ]:
!wc -l {DATA_ROOT}/manifests/dev_test.jsonl
!head -1 {DATA_ROOT}/manifests/dev_test.jsonl

If the sample count above is 0, the dyadic filters (exactly 2 speakers,
listener visibility >= 70%, target speech >= 1s -- see
`preprocessing/build_meld_dyadic_subset.py`) are dropping everything in
your test slice. Try a larger `--limit_dialogues` or inspect a few
dialogues manually before scaling up.

Once satisfied, run over the full dev split (drop `--limit_dialogues`) and
split part of it off for validation:

In [ ]:
!python -m preprocessing.build_meld_dyadic_subset \
    --meld_csv {MELD_CSV} \
    --video_dir {MELD_VIDEO_DIR} \
    --out_manifest {DATA_ROOT}/manifests/dev_full.jsonl

In [ ]:
# quick 90/10 train/val split of the resulting manifest
import json, random
with open(f'{DATA_ROOT}/manifests/dev_full.jsonl') as f:
    lines = [l for l in f if l.strip()]
random.Random(42).shuffle(lines)
n_val = max(1, len(lines) // 10)
val_lines, train_lines = lines[:n_val], lines[n_val:]
with open(f'{DATA_ROOT}/manifests/train.jsonl', 'w') as f:
    f.writelines(train_lines)
with open(f'{DATA_ROOT}/manifests/val.jsonl', 'w') as f:
    f.writelines(val_lines)
print(f'train: {len(train_lines)}  val: {len(val_lines)}')

## 6. Point Stage A's config at the real manifests and train

In [ ]:
from react_tts.config import load_config
import yaml

cfg = load_config('configs/stage_a.yaml')
cfg['data']['manifest_train'] = f'{DATA_ROOT}/manifests/train.jsonl'
cfg['data']['manifest_val'] = f'{DATA_ROOT}/manifests/val.jsonl'
cfg['data']['synthetic'] = False
cfg['train']['ckpt_dir'] = f'{DATA_ROOT}/checkpoints/stage_a'
with open('configs/stage_a_meld.yaml', 'w') as f:
    yaml.safe_dump(dict(cfg), f)
print(open('configs/stage_a_meld.yaml').read())

In [ ]:
!python -m scripts.train_stage_a --config configs/stage_a_meld.yaml --device cuda

## 7. RQ3 sanity check: emotion-mirroring baseline (no training)
Compares against the learned planner above -- see README RQ3 / design doc §13.

In [ ]:
!python -m scripts.train_stage_a --config configs/stage_a_meld.yaml --mode mirror

## Next steps
- Ablations (same command, one flag each): `--no_listener_face`, `--static_face`,
  `--no_reaction_delta`, `--random_listener`.
- Stage B (expressive TTS backbone) needs its own audio data (e.g. CREMA-D) --
  see the main README's "Data strategy".
- Checkpoints land in `{DATA_ROOT}/checkpoints/stage_a/` on Drive, so they
  survive session restarts.